# Manuscript Figures

Renders the spike manuscript figure surface from the CSVs exported by
`evaluate` and `cross_validation`.

This notebook reads **CSVs only**. It must never load `fit_collection.pkl`
(1.76 GB) — everything it needs was exported by `evaluate` for exactly that
reason.

**Figures produced**

| Fig | File |
|-----|------|
| S6  | `raw_data_summary_barcodes_backgrounds_hist` |
| S7  | `replicate_functional_score_correlation_scatter` |
| S9  | `shrinkage_analysis_trace_plots_beta` |
| S11 | `percent_shifts_under_x_lineplot` |
| S12 | `shift_corr_Delta_BA2` |
| S16 | `convergence_all_lasso_lines` |
| S17 | `global_epistasis_and_prediction_correlations` |
| 4   | `shift_by_site_heatmap_zoom` |
| 5   | `validation_titer_fold_change` |

In [ ]:
import warnings

warnings.filterwarnings("ignore")

import os
import sys

sys.path.insert(0, "notebooks")

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

from _common import load_config
from _downstream import (
    fetch_validation_data,
    lasso_slice,
    savefig,
    set_plot_style,
)

In [ ]:
config_path = "config/config.yaml"
downstream_config_path = "config/config_downstream.yaml"
output_dir = None

In [ ]:
config = load_config(config_path, downstream_config_path)
spike = config["spike"]
lasso_choice = spike["lasso_choice"]
condition_titles = spike["condition_titles"]
condition_colors = spike["condition_colors"]
domain_dict = spike["domain_dict"]

fig_cfg = spike["figures"]
FORMATS = tuple(fig_cfg["formats"])
DPI = fig_cfg["dpi"]
EXCLUDED_FUSIONREG = fig_cfg["excluded_fusionreg"]
heatmap_cfg = fig_cfg["heatmap"]
validation_cfg = fig_cfg["validation"]

if output_dir is None:
    output_dir = spike.get("output_dir", "results")

FIGURES_DIR = os.path.join(output_dir, "figures")
set_plot_style()

# Conditions in manuscript order. Omicron_BA1 is the reference condition, so
# it has no shift column -- shift figures use the other two.
REFERENCE = "Omicron_BA1"
CONDITIONS = ["Delta", "Omicron_BA1", "Omicron_BA2"]
SHIFT_CONDITIONS = [c for c in CONDITIONS if c != REFERENCE]

print(f"lasso_choice = {lasso_choice:g}")
print(f"figures -> {FIGURES_DIR}")
print(f"excluded fusionreg rungs: {EXCLUDED_FUSIONREG or 'none'}")

In [ ]:
def read(name):
    """Read an exported CSV from the pipeline results directory."""
    return pd.read_csv(os.path.join(output_dir, name))


func_score_df = read("training_functional_scores.csv").fillna(
    {"aa_substitutions": ""}
)
mutations_df = read("mutations_df.csv")
collection_muts = read("collection_muts.csv")
fit_sparsity = read("fit_sparsity.csv")
replicate_corr = read("library_replicate_correlation.csv")
convergence_trajectory = read("convergence_trajectory.csv")
ge_variants = read("ge_landscape_variants.csv")
ge_curve = read("ge_landscape_curve.csv")
ge_params = read("ge_params.csv")
cv_loss = read("cross_validation_loss.csv")

print(f"func_score_df          {func_score_df.shape}")
print(f"mutations_df           {mutations_df.shape}")
print(f"collection_muts        {collection_muts.shape}")
print(f"convergence_trajectory {convergence_trajectory.shape}")

In [ ]:
def drop_excluded(df, column="fusionreg"):
    """Remove regularization rungs excluded from analysis.

    The top rung of the prod ladder is retained in the fit (removing it would
    force a full refit) but is unstable -- one replicate diverges after ~100
    sweeps. Figures that sweep the ladder must not present it as a normal
    point. See ``spike.figures.excluded_fusionreg``.
    """
    if not EXCLUDED_FUSIONREG:
        return df
    return df[~df[column].isin(EXCLUDED_FUSIONREG)]


LADDER = sorted(drop_excluded(collection_muts)["fusionreg"].unique())
print(f"analysed ladder ({len(LADDER)} rungs): {[f'{x:g}' for x in LADDER]}")

## Figure S6 — variant counts per background

Barcode (variant) count distributions per background and replicate. Reads
the post-filter training set, so counts reflect what the model actually saw.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.6))

# Panel A: variants per condition/replicate.
counts = (
    func_score_df.groupby(["condition", "replicate"]).size().reset_index(name="n")
)
width = 0.38
x = np.arange(len(CONDITIONS))
replicates = sorted(counts["replicate"].unique())
for i, rep in enumerate(replicates):
    sub = counts[counts["replicate"] == rep].set_index("condition")
    heights = [sub.loc[c, "n"] for c in CONDITIONS]
    axes[0].bar(
        x + (i - 0.5) * width,
        heights,
        width,
        color=[condition_colors[c] for c in CONDITIONS],
        edgecolor="black",
        linewidth=0.5,
        alpha=1.0 if i == 0 else 0.55,
    )
axes[0].set_xticks(x)
axes[0].set_xticklabels([condition_titles[c] for c in CONDITIONS])
axes[0].set_ylabel("variants")
axes[0].set_title("A. Variants per background", loc="left")
# Bars are colored by CONDITION, so a colored legend swatch would imply the
# color encodes replicate. Use neutral grey patches keyed to the alpha
# difference, which is what actually distinguishes the pairs.
axes[0].legend(
    handles=[
        patches.Patch(
            facecolor="#777777",
            alpha=1.0 if i == 0 else 0.55,
            edgecolor="black",
            linewidth=0.5,
            label=f"replicate {rep}",
        )
        for i, rep in enumerate(replicates)
    ],
    frameon=False,
)

# Panel B: distribution of mutations per variant.
for c in CONDITIONS:
    sub = func_score_df[func_score_df["condition"] == c]
    axes[1].hist(
        sub["n_subs"],
        bins=np.arange(0, sub["n_subs"].max() + 2) - 0.5,
        histtype="step",
        linewidth=1.4,
        color=condition_colors[c],
        label=condition_titles[c],
        density=True,
    )
axes[1].set_xlabel("amino-acid substitutions per variant")
axes[1].set_ylabel("density")
axes[1].set_title("B. Mutations per variant", loc="left")
axes[1].legend(frameon=False)
axes[1].set_xlim(-0.5, 15)

fig.tight_layout()
savefig(fig, "raw_data_summary_barcodes_backgrounds_hist", FIGURES_DIR, FORMATS, DPI)
plt.show()

## Figure S7 — replicate functional-score correlation

Per-background scatter of functional scores for variants observed in both
replicate libraries, with Pearson r.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(7.5, 2.7), sharex=True, sharey=True)

for ax, cond in zip(axes, CONDITIONS):
    sub = func_score_df[func_score_df["condition"] == cond]
    wide = sub.pivot_table(
        index="aa_substitutions", columns="replicate", values="func_score"
    ).dropna()
    reps = sorted(wide.columns)
    xv, yv = wide[reps[0]].to_numpy(), wide[reps[1]].to_numpy()
    ax.scatter(xv, yv, s=2, alpha=0.25, color=condition_colors[cond], linewidths=0)
    r = pearsonr(xv, yv)[0]
    ax.set_title(f"{condition_titles[cond]}\nr = {r:.2f}  (n = {len(wide):,})", loc="left")
    ax.set_xlabel(f"replicate {reps[0]}")
    lo = float(np.nanpercentile(np.concatenate([xv, yv]), 0.5))
    hi = float(np.nanpercentile(np.concatenate([xv, yv]), 99.5))
    ax.plot([lo, hi], [lo, hi], ls="--", lw=0.8, color="black", zorder=3)
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)

axes[0].set_ylabel("replicate 2 functional score")
fig.tight_layout()
savefig(
    fig,
    "replicate_functional_score_correlation_scatter",
    FIGURES_DIR,
    FORMATS,
    DPI,
)
plt.show()

## Figure S16 — convergence across the lasso ladder

The outer convergence criterion is a *between-sweep relative change* in the
objective, not a gradient norm:

$$\text{objective\_error} = \frac{|f_{k-1} - f_k|}{\max(|f_{k-1}|, |f_k|, 1)}$$

compared against `tol`. One line per (replicate, lasso weight).

In [ ]:
traj = drop_excluded(convergence_trajectory)
tol = spike["fitting"]["tol"]

fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.9), sharex=True)
datasets = sorted(traj["dataset_name"].unique())
cmap = plt.get_cmap("viridis")
norm = plt.Normalize(0, max(len(LADDER) - 1, 1))

for ax, ds in zip(axes, datasets):
    sub = traj[traj["dataset_name"] == ds]
    for i, fr in enumerate(LADDER):
        s = sub[sub["fusionreg"] == fr].sort_values("iteration")
        if s.empty:
            continue
        ax.semilogy(
            s["iteration"],
            s["objective_error_trajectory"].clip(lower=1e-12),
            lw=1.0,
            color=cmap(norm(i)),
            label=f"{fr:g}",
        )
    ax.axhline(tol, ls="--", lw=1.0, color="red")
    ax.text(
        0.98,
        tol,
        f"  tol = {tol:g}",
        transform=ax.get_yaxis_transform(),
        ha="right",
        va="bottom",
        fontsize=6,
        color="red",
    )
    ax.set_title(ds, loc="left")
    ax.set_xlabel("block-coordinate sweep")

axes[0].set_ylabel("objective error\n(relative change per sweep)")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    title="fusionreg",
    frameon=False,
    loc="center left",
    bbox_to_anchor=(1.0, 0.5),
    fontsize=6,
    title_fontsize=7,
)
fig.tight_layout()
savefig(fig, "convergence_all_lasso_lines", FIGURES_DIR, FORMATS, DPI)
plt.show()

## Figure S9 — shrinkage analysis

Three panels sweeping the lasso ladder: replicate correlation of parameters,
held-out loss, and shift sparsity by mutation class. Together these are the
three criteria used to choose λ.

`cross_validation_loss` values are **already per-variant averages** (the
loss uses `.mean()`); do not divide by variant count again.

In [ ]:
# The ladder includes lambda = 0 (the unregularized anchor), so the x-axis
# must be symlog rather than log. linthresh sits strictly BELOW the smallest
# nonzero rung: setting it equal to that rung folds 0 into the linear region
# and matplotlib then draws a colliding "0"/"1e-6" tick pair and a spurious
# negative decade. Only the zero tick plus the real rungs are labelled.
_nonzero = [x for x in LADDER if x > 0]
LINTHRESH = min(_nonzero) / 2 if _nonzero else 1e-6


def style_lambda_axis(ax):
    """Apply the shared symlog lambda axis, showing zero without a fake decade."""
    ax.set_xscale("symlog", linthresh=LINTHRESH, linscale=0.5)
    # Clamp to the data. symlog's default upper limit rounds up to the next
    # decade (10^0 here), which would leave most of the panel empty since the
    # ladder tops out below 10^-3.
    ax.set_xlim(0, max(LADDER) * 1.3)
    ax.set_xlabel("fusionreg (λ)")
    # Label 0 explicitly and drop the decade tick nearest it: symlog places a
    # minor decade inside the linear region where it visually collides with
    # the zero tick.
    decades = [
        t
        for t in ax.get_xticks()
        if LINTHRESH * 2 <= t <= max(LADDER) * 1.3
    ]
    ax.set_xticks([0] + decades)
    ax.set_xticklabels(["0"] + [f"$10^{{{int(np.log10(t))}}}$" for t in decades])


fig, axes = plt.subplots(1, 3, figsize=(7.6, 3.1))

# Panel A: replicate correlation of beta and shift parameters.
corr = drop_excluded(replicate_corr)
for param in ["beta_Delta", "beta_Omicron_BA1", "beta_Omicron_BA2"]:
    s = corr[corr["mut_param"] == param].sort_values("fusionreg")
    cond = param.replace("beta_", "")
    axes[0].plot(
        s["fusionreg"],
        s["correlation"],
        marker="o",
        ms=3,
        lw=1.2,
        color=condition_colors[cond],
        label=rf"$\beta$ {condition_titles[cond]}",
    )
for param in ["shift_Delta", "shift_Omicron_BA2"]:
    s = corr[corr["mut_param"] == param].sort_values("fusionreg")
    cond = param.replace("shift_", "")
    axes[0].plot(
        s["fusionreg"],
        s["correlation"],
        marker="s",
        ms=3,
        lw=1.2,
        ls="--",
        color=condition_colors[cond],
        label=rf"$\Delta$ {condition_titles[cond]}",
    )
style_lambda_axis(axes[0])
axes[0].set_ylabel("replicate correlation")
axes[0].set_title("A. Parameter reproducibility", loc="left")
# Legend below the axes: the beta BA.2 trace runs through the lower-left
# corner, where an inset legend would sit on top of it.
axes[0].legend(
    frameon=False,
    fontsize=5.5,
    ncol=2,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.28),
)

# Panel B: cross-validation loss.
cvl = drop_excluded(cv_loss)
for split, ls in [("training", "--"), ("validation", "-")]:
    s = (
        cvl[cvl["dataset"] == split]
        .groupby("fusionreg", as_index=False)["mean_loss"]
        .mean()
        .sort_values("fusionreg")
    )
    axes[1].plot(
        s["fusionreg"], s["mean_loss"], marker="o", ms=3, lw=1.2, ls=ls, label=split
    )
axes[1].axvline(lasso_choice, color="red", lw=1.0, ls=":")
style_lambda_axis(axes[1])
axes[1].set_ylabel("loss per variant")
axes[1].set_title("B. Held-out loss", loc="left")
axes[1].legend(frameon=False)

# Panel C: shift sparsity, stop vs nonsynonymous.
sp = drop_excluded(fit_sparsity)
for mut_type, ls in [("stop", "-"), ("nonsynonymous", "--")]:
    s = (
        sp[sp["mut_type"] == mut_type]
        .groupby("fusionreg", as_index=False)["sparsity"]
        .mean()
        .sort_values("fusionreg")
    )
    axes[2].plot(
        s["fusionreg"],
        s["sparsity"],
        marker="o",
        ms=3,
        lw=1.2,
        ls=ls,
        label=mut_type,
    )
axes[2].axvline(lasso_choice, color="red", lw=1.0, ls=":")
style_lambda_axis(axes[2])
axes[2].set_ylabel("fraction of shifts = 0")
axes[2].set_title("C. Shift sparsity", loc="left")
axes[2].legend(frameon=False)

fig.tight_layout()
savefig(fig, "shrinkage_analysis_trace_plots_beta", FIGURES_DIR, FORMATS, DPI)
plt.show()

## Figure S17 — global epistasis and prediction correlations

The fitted sigmoid mapping latent phenotype to functional score, plus the
correlation between predicted and measured functional scores.

In [ ]:
# One column per condition. Overlaying all three on shared axes is what the
# legacy figure did, but with ~200k variants the last-drawn condition hides
# the other two completely -- the per-condition r values in the legend then
# describe points the reader cannot see. Faceting keeps every condition
# visible; the shared GE curve is repeated in each panel for comparison.
first_ds = sorted(ge_variants["dataset_name"].unique())[0]
gv = ge_variants[ge_variants["dataset_name"] == first_ds]
gc = ge_curve[ge_curve["dataset_name"] == first_ds].sort_values("predicted_latent")

fig, axes = plt.subplots(2, len(CONDITIONS), figsize=(7.4, 4.8))

lims = [
    float(np.nanpercentile(gv["func_score"], 0.5)),
    float(np.nanpercentile(gv["func_score"], 99.5)),
]

for j, cond in enumerate(CONDITIONS):
    s = gv[gv["condition"] == cond]

    # Row 1: latent phenotype vs measured functional score, with the GE curve.
    ax = axes[0, j]
    ax.scatter(
        s["predicted_latent"],
        s["func_score"],
        s=1.0,
        alpha=0.12,
        color=condition_colors[cond],
        linewidths=0,
        rasterized=True,
    )
    ax.plot(gc["predicted_latent"], gc["ge_curve_value"], color="black", lw=1.4)
    ax.set_title(condition_titles[cond], loc="left")
    ax.set_xlabel("latent phenotype")
    if j == 0:
        ax.set_ylabel("measured\nfunctional score")

    # Row 2: predicted vs measured, with Pearson r.
    ax = axes[1, j]
    r = pearsonr(s["predicted_func_score"], s["func_score"])[0]
    ax.scatter(
        s["predicted_func_score"],
        s["func_score"],
        s=1.0,
        alpha=0.12,
        color=condition_colors[cond],
        linewidths=0,
        rasterized=True,
    )
    ax.plot(lims, lims, ls="--", lw=0.8, color="black")
    ax.set_title(f"r = {r:.2f}  (n = {len(s):,})", loc="left")
    ax.set_xlabel("predicted functional score")
    ax.set_xlim(*lims)
    ax.set_ylim(*lims)
    if j == 0:
        ax.set_ylabel("measured\nfunctional score")

axes[0, 0].get_figure().suptitle(
    f"A. Global epistasis   |   B. Prediction accuracy   ({first_ds})",
    x=0.01,
    ha="left",
    fontsize=9,
)
fig.tight_layout(rect=[0, 0, 1, 0.96])
savefig(
    fig, "global_epistasis_and_prediction_correlations", FIGURES_DIR, FORMATS, DPI
)
plt.show()

## Figure S11 — cumulative distribution of shift magnitudes

Fraction of mutations whose absolute shift falls under a threshold, at the
chosen λ. The steep rise near zero is the lasso doing its job.

In [ ]:
muts_at_lambda = lasso_slice(collection_muts, lasso_choice)

fig, ax = plt.subplots(figsize=(3.6, 2.8))
thresholds = np.linspace(0, 2.0, 200)

for cond in SHIFT_CONDITIONS:
    col = f"shift_{cond}"
    for ds, ls in zip(sorted(muts_at_lambda["dataset_name"].unique()), ["-", "--"]):
        vals = muts_at_lambda.loc[
            muts_at_lambda["dataset_name"] == ds, col
        ].abs().dropna()
        frac = [(vals <= t).mean() * 100 for t in thresholds]
        ax.plot(
            thresholds,
            frac,
            lw=1.3,
            ls=ls,
            color=condition_colors[cond],
            label=f"{condition_titles[cond]} ({ds})",
        )

ax.set_xlabel("|shift| threshold")
ax.set_ylabel("% of mutations under threshold")
ax.set_title(f"λ = {lasso_choice:g}", loc="left")
ax.legend(frameon=False, fontsize=6)
ax.set_ylim(0, 101)
fig.tight_layout()
savefig(fig, "percent_shifts_under_x_lineplot", FIGURES_DIR, FORMATS, DPI)
plt.show()

## Figure S12 — Delta vs BA.2 shift correlation

Are the condition-specific shifts shared between the two non-reference
backgrounds, or background-specific? Uses replicate-averaged shifts.

In [ ]:
fig, ax = plt.subplots(figsize=(3.4, 3.2))

sub = mutations_df.dropna(subset=["avg_shift_Delta", "avg_shift_Omicron_BA2"])
nonsyn = sub[sub["sense"] == "nonsynonymous"]
stop = sub[sub["sense"] == "stop"]

ax.scatter(
    nonsyn["avg_shift_Delta"],
    nonsyn["avg_shift_Omicron_BA2"],
    s=4,
    alpha=0.35,
    color="#444444",
    linewidths=0,
    label="nonsynonymous",
)
ax.scatter(
    stop["avg_shift_Delta"],
    stop["avg_shift_Omicron_BA2"],
    s=8,
    alpha=0.8,
    color="red",
    marker="x",
    linewidths=0.7,
    label="stop",
)

r = pearsonr(sub["avg_shift_Delta"], sub["avg_shift_Omicron_BA2"])[0]
ax.axhline(0, lw=0.6, color="black", zorder=0)
ax.axvline(0, lw=0.6, color="black", zorder=0)
ax.set_xlabel(f"shift, {condition_titles['Delta']}")
ax.set_ylabel(f"shift, {condition_titles['Omicron_BA2']}")
ax.set_title(f"r = {r:.2f}  (n = {len(sub):,})", loc="left")
ax.legend(frameon=False, fontsize=6)
fig.tight_layout()
savefig(fig, "shift_corr_Delta_BA2", FIGURES_DIR, FORMATS, DPI)
plt.show()

## Figure 5 — model prediction vs measured viral titer

The five validation mutations were introduced into pseudoviruses and their
titers measured, giving an independent test of the model's predictions.

**The x-axis is neither β nor a shift.** It is the predicted *enrichment
ratio*, `2 ** predicted_func_score_{condition}` — the legacy variable was
named `predicted_beta`, which is misleading. A reviewer asked about this
explicitly; it is answered in the response-to-reviews.

`spike_validation_data.csv` already holds titer fold-changes relative to the
unmutated background, so it is used directly rather than re-derived from
`viral_titers.csv`.

In [ ]:
titers, validation = fetch_validation_data(output_dir)

VALIDATION_MUTATIONS = validation_cfg["mutations"]
BACKGROUND_TO_CONDITION = validation_cfg["background_to_condition"]

# The titer file labels backgrounds "Delta"/"BA.1"/"BA.2" while the pipeline
# uses "Delta"/"Omicron_BA1"/"Omicron_BA2". Mapping is mandatory: joining on
# the raw labels silently yields an empty frame for the two Omicron rows.
measured = validation.melt(
    id_vars=["background", "replicate"],
    value_vars=VALIDATION_MUTATIONS,
    var_name="mutation",
    value_name="titer_fold_change",
)
measured["condition"] = measured["background"].map(BACKGROUND_TO_CONDITION)
unmapped = measured[measured["condition"].isna()]["background"].unique()
if len(unmapped):
    raise ValueError(
        f"Validation backgrounds {sorted(unmapped)} have no entry in "
        "spike.figures.validation.background_to_condition. Without a mapping "
        "these rows would silently drop out of the Figure 5 join."
    )

# Predicted enrichment ratio at the chosen lambda, averaged over replicates.
pred_long = []
for cond in CONDITIONS:
    col = f"predicted_func_score_{cond}"
    s = (
        muts_at_lambda[muts_at_lambda["mutation"].isin(VALIDATION_MUTATIONS)]
        .groupby("mutation", as_index=False)[col]
        .mean()
        .rename(columns={col: "predicted_func_score"})
    )
    s["condition"] = cond
    pred_long.append(s)
predicted = pd.concat(pred_long, ignore_index=True)
predicted["enrichment_ratio"] = 2 ** predicted["predicted_func_score"]

fig5 = measured.merge(predicted, on=["mutation", "condition"], how="left")

# A validation mutation can legitimately be absent: the test profile
# subsamples variants (subsample_frac), so a given mutation may never be
# observed. Drop those panels with a loud warning rather than failing --
# but on the full data this should never trigger, and an empty result still
# raises, because that means the join itself is broken (the usual cause is
# a background-name mismatch, which background_to_condition guards above).
missing = fig5[fig5["enrichment_ratio"].isna()]
if len(missing):
    absent = sorted(missing["mutation"].unique())
    print(
        f"  WARNING: no model prediction at lambda={lasso_choice:g} for "
        f"{absent}. These mutations are not in collection_muts.csv -- "
        "expected when the profile subsamples variants, unexpected on full "
        "data. Their panels are omitted."
    )
    fig5 = fig5.dropna(subset=["enrichment_ratio"])

if fig5.empty:
    raise ValueError(
        "The Figure 5 join produced no rows. Every validation mutation "
        "failed to match a model prediction, which usually means the "
        "measured/predicted key spaces disagree rather than that data is "
        f"missing. lambda={lasso_choice:g}; "
        f"validation mutations={VALIDATION_MUTATIONS}."
    )

plotted_mutations = [m for m in VALIDATION_MUTATIONS if m in set(fig5["mutation"])]
n_mut = len(plotted_mutations)
fig, axes = plt.subplots(1, n_mut, figsize=(1.55 * n_mut, 2.4), sharex=True, sharey=True)

for ax, mut in zip(np.atleast_1d(axes), plotted_mutations):
    sub = fig5[fig5["mutation"] == mut]
    for cond in CONDITIONS:
        s = sub[sub["condition"] == cond]
        ax.scatter(
            s["enrichment_ratio"],
            s["titer_fold_change"],
            s=22,
            color=condition_colors[cond],
            edgecolor="black",
            linewidth=0.4,
            label=condition_titles[cond],
            zorder=3,
        )
    ax.set_yscale("log")
    ax.set_xlim(*validation_cfg["xlim"])
    ax.set_ylim(*validation_cfg["ylim"])
    ax.set_title(mut, loc="left")
    ax.set_xlabel("predicted\nenrichment ratio")

np.atleast_1d(axes)[0].set_ylabel("measured titer\nfold change")
handles, labels = np.atleast_1d(axes)[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    frameon=False,
    loc="center left",
    bbox_to_anchor=(1.0, 0.5),
    fontsize=6,
)
fig.tight_layout()
savefig(fig, "validation_titer_fold_change", FIGURES_DIR, FORMATS, DPI)
plt.show()

# Headline claim check (manuscript main.tex): A419S is near-neutral in Delta
# but abolishes titer in BA.1/BA.2. Reported, not asserted -- if the model's
# contrast ever collapses, the number below shows it rather than the figure
# quietly misrepresenting it.
a419s = predicted[predicted["mutation"] == "A419S"].dropna(
    subset=["enrichment_ratio"]
).set_index("condition")
if set(CONDITIONS).issubset(a419s.index):
    delta_er = a419s.loc["Delta", "enrichment_ratio"]
    print("A419S predicted enrichment ratio:")
    for cond in CONDITIONS:
        er = a419s.loc[cond, "enrichment_ratio"]
        fold = delta_er / er if er else float("inf")
        print(f"  {condition_titles[cond]:>6}: {er:.3f}   (Delta/{condition_titles[cond]} = {fold:.1f}x)")
else:
    print("A419S not present in all conditions -- headline check skipped "
          "(expected on subsampled profiles).")

## Figure 4 — shift by site, with zoom panels

Top: shift magnitude along the spike primary sequence with domain
architecture. Bottom: per-site amino-acid heatmaps for five zoom windows.

BA.1 is the reference condition, so only Delta and BA.2 have shifts.

In [ ]:
site_ranges = heatmap_cfg["site_ranges"]
AA_ORDER = heatmap_cfg["aa_order"]

# The zoom windows were chosen by eye from v0.4.0 shift values. lambda is
# unchanged at 8.0e-05, so they should still hold -- but verify rather than
# assume.
#
# Rank PER CONDITION, not on a pooled max. Delta's shifts are globally larger
# than BA.2's, so a pooled ranking buries any BA.2-specific window. zoom1 is
# exactly that case: site 142 carries D142L, one of the five experimentally
# validated mutations, with a BA.2 shift of -0.96 (rank ~6 within BA.2) but a
# Delta shift of only -0.11. A pooled metric ranks it ~131st and reads as
# "stale window" when the window is in fact validation-motivated.
#
# So: this is a report, not a gate. Windows are chosen for biological
# relevance -- shift rank is corroboration, not the selection criterion.
site_ranks = {}
for cond in SHIFT_CONDITIONS:
    s = (
        mutations_df.groupby("sites", as_index=False)[f"avg_shift_{cond}"]
        .apply(lambda x: x.abs().max())
        .rename(columns={f"avg_shift_{cond}": "abs_shift"})
        .sort_values("abs_shift", ascending=False)
        .reset_index(drop=True)
    )
    s["rank"] = s.index + 1
    site_ranks[cond] = s.set_index("sites")

n_sites = len(site_ranks[SHIFT_CONDITIONS[0]])
print(f"Zoom-window check -- strongest site per window, ranked within each "
      f"condition (of {n_sites} sites):")
for name, (lo, hi) in site_ranges.items():
    parts = []
    for cond in SHIFT_CONDITIONS:
        s = site_ranks[cond]
        win = s[(s.index >= lo) & (s.index <= hi)]
        if win.empty:
            parts.append(f"{condition_titles[cond]}: no data")
            continue
        best_site = win["abs_shift"].idxmax()
        parts.append(
            f"{condition_titles[cond]}: site {int(best_site)} "
            f"|shift|={win.loc[best_site, 'abs_shift']:.2f} "
            f"(rank {int(win.loc[best_site, 'rank'])})"
        )
    print(f"  {name} [{lo}-{hi}]  " + " | ".join(parts))

validated_sites = {
    int("".join(ch for ch in m if ch.isdigit())): m
    for m in VALIDATION_MUTATIONS
}
for name, (lo, hi) in site_ranges.items():
    hits = [m for site, m in validated_sites.items() if lo <= site <= hi]
    if hits:
        print(f"  {name} contains validated mutation(s): {', '.join(hits)}")

In [ ]:
zoom_names = list(site_ranges.keys())
widths = [site_ranges[z][1] - site_ranges[z][0] + 1 for z in zoom_names]
# First column is inflated to make room for the colorbar and AA tick labels.
widths[0] *= 1.5

mosaic = [
    ["domain"] * len(zoom_names),
    ["Delta_line"] * len(zoom_names),
    ["BA2_line"] * len(zoom_names),
    [f"Delta_{z}" for z in zoom_names],
    [f"BA2_{z}" for z in zoom_names],
]
# The heatmap rows carry 22 amino-acid tick labels each, so they need enough
# height that the labels do not overlap. A taller figure with generous
# height_ratios on the two heatmap rows is what buys that room; shrinking the
# font instead makes the residue labels unreadable at print size.
fig, axs = plt.subplot_mosaic(
    mosaic,
    figsize=(7.2, 10.4),
    height_ratios=[0.45, 1.25, 1.25, 3.3, 3.3],
    width_ratios=widths,
    gridspec_kw={
        "hspace": 0.45,
        "wspace": 0.12,
        # Reserve the bottom strip for the colorbar axes added below.
        "bottom": 0.10,
        "top": 0.96,
        "left": 0.085,
        "right": 0.98,
    },
)

SITE_MIN = int(mutations_df["sites"].min())
SITE_MAX = int(mutations_df["sites"].max())

# --- Domain architecture ribbon ---
ax = axs["domain"]
ax.set_xlim(SITE_MIN, SITE_MAX)
ax.set_ylim(0, 1)
ax.axhspan(0.35, 0.65, color="#EEEEEE", zorder=0)
for i, (dom, (lo, hi)) in enumerate(domain_dict.items()):
    ax.add_patch(
        patches.Rectangle(
            (lo, 0.25),
            hi - lo,
            0.5,
            facecolor=plt.get_cmap("tab20")(i % 20),
            edgecolor="black",
            linewidth=0.4,
        )
    )
    ax.text(
        (lo + hi) / 2,
        0.9 if i % 2 else 0.02,
        dom,
        ha="center",
        va="bottom" if i % 2 else "top",
        fontsize=5.5,
    )
ax.set_yticks([])
ax.set_xticks([])
for side in ["top", "right", "left", "bottom"]:
    ax.spines[side].set_visible(False)
ax.set_title("Spike domain architecture", loc="left", fontsize=8)

# --- Per-site shift scatter, one row per non-reference condition ---
line_axes = {"Delta": axs["Delta_line"], "Omicron_BA2": axs["BA2_line"]}
for cond, ax in line_axes.items():
    per_site = (
        mutations_df.groupby("sites", as_index=False)[f"avg_shift_{cond}"]
        .apply(lambda s: s.abs().max())
        .rename(columns={f"avg_shift_{cond}": "max_abs_shift"})
    )
    ax.scatter(
        per_site["sites"],
        per_site["max_abs_shift"],
        s=3,
        color=condition_colors[cond],
        linewidths=0,
    )
    for lo, hi in site_ranges.values():
        ax.axvspan(lo, hi, color="black", alpha=0.10, zorder=0)
    ax.set_xlim(SITE_MIN, SITE_MAX)
    ax.set_ylim(0, heatmap_cfg["scatter_ylim"][1])
    ax.set_ylabel(f"|shift|\n{condition_titles[cond]}")
    ax.set_xlabel("site")

# --- Zoom heatmaps ---
vmin, vmax = heatmap_cfg["vmin"], heatmap_cfg["vmax"]
# RdBu runs red->blue as the value INCREASES, which would paint positive
# shifts blue. Reverse it so the usual convention holds: red = positive
# (shift toward higher function), blue = negative.
cmap = plt.get_cmap(heatmap_cfg["cmap"] + "_r").copy()
cmap.set_bad(heatmap_cfg["missing_color"])

mesh = None
for cond, prefix in [("Delta", "Delta"), ("Omicron_BA2", "BA2")]:
    for zi, zname in enumerate(zoom_names):
        lo, hi = site_ranges[zname]
        ax = axs[f"{prefix}_{zname}"]
        sites = list(range(lo, hi + 1))
        grid = pd.DataFrame(index=AA_ORDER, columns=sites, dtype=float)
        sub = mutations_df[mutations_df["sites"].isin(sites)]
        for _, row in sub.iterrows():
            if row["muts"] in grid.index:
                grid.loc[row["muts"], row["sites"]] = row[f"avg_shift_{cond}"]

        mesh = ax.pcolormesh(
            np.arange(len(sites) + 1),
            np.arange(len(AA_ORDER) + 1),
            np.ma.masked_invalid(grid.to_numpy(dtype=float)),
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
            edgecolors="white",
            linewidth=0.15,
        )

        # Mark the wildtype residue at each site with a black x.
        wt_map = sub.drop_duplicates("sites").set_index("sites")["wts"].to_dict()
        for si, site in enumerate(sites):
            wt = wt_map.get(site)
            if wt in AA_ORDER:
                ax.plot(
                    si + 0.5,
                    AA_ORDER.index(wt) + 0.5,
                    marker="x",
                    color="black",
                    ms=3,
                    mew=0.8,
                )

        ax.set_xticks(np.arange(len(sites)) + 0.5)
        ax.set_xticklabels(sites, fontsize=5, rotation=90)
        ax.set_yticks(np.arange(len(AA_ORDER)) + 0.5)
        ax.set_yticklabels(AA_ORDER if zi == 0 else [], fontsize=5.5)
        ax.invert_yaxis()
        ax.tick_params(length=0)
        for side in ["top", "right", "left", "bottom"]:
            ax.spines[side].set_visible(False)
        if zi == 0:
            ax.set_ylabel(condition_titles[cond], fontsize=7)

# Place the colorbar in its own explicit axes rather than stealing space from
# the mosaic panels: `ax=[...]` shrinks the donor axes, which desynchronizes
# the zoom columns from the site-scatter rows above them and leaves the bar
# hanging under the widest column instead of centered.
cbar_ax = fig.add_axes([0.30, 0.045, 0.40, 0.012])
cbar = fig.colorbar(mesh, cax=cbar_ax, orientation="horizontal")
cbar.set_label("shift (red = positive, blue = negative)", fontsize=7)
cbar.ax.tick_params(labelsize=6)

savefig(fig, "shift_by_site_heatmap_zoom", FIGURES_DIR, FORMATS, DPI)
plt.show()